# 09 — Decision Threshold Optimization

This notebook optimizes the classification decision threshold for severe-rainfall forecasting without using the final 2021 test period for model selection.

The chronological strategy is:

- 2018–2019: model training;
- 2020: decision-threshold validation;
- 2021: final unseen test evaluation.

Logistic Regression and Random Forest are evaluated because both demonstrated probability-ranking ability during the baseline experiments.

For each forecasting horizon, the decision threshold is selected on the 2020 validation period by maximizing F1-score.

After threshold selection, each model is retrained using the complete 2018–2020 training period and evaluated once on the untouched 2021 test period.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone

from sklearn.ensemble import (
    RandomForestClassifier,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
)

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_PATH = Path(
    "../data/processed/recife_2018_2021_model_dataset.csv"
)

OUTPUT_PATH = Path(
    "../data/processed/threshold_optimized_results.csv"
)

RANDOM_STATE = 42

VALIDATION_START = pd.Timestamp(
    "2020-01-01 00:00",
    tz="UTC",
)

TEST_START = pd.Timestamp(
    "2021-01-01 00:00",
    tz="UTC",
)

FORECAST_HORIZONS = [
    1,
    2,
    3,
    4,
    5,
    6,
]


# ============================================================
# 2. FEATURE SET
# ============================================================

FEATURE_COLUMNS = [
    "precipitation_mm",
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",

    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",

    "precipitation_lag_1h",
    "precipitation_lag_2h",
    "precipitation_lag_3h",
    "precipitation_lag_6h",
    "precipitation_sum_3h",
    "precipitation_sum_6h",

    "temperature_c_lag_1h",
    "humidity_pct_lag_1h",
    "pressure_station_mb_lag_1h",

    "temperature_change_1h",
    "humidity_change_1h",
    "pressure_change_1h",
]


# ============================================================
# 3. LOAD DATA
# ============================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_PATH}"
    )


df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["datetime_utc"],
)

df = (
    df
    .set_index("datetime_utc")
    .sort_index()
)


print("Dataset loaded.")
print("Rows:", len(df))


# ============================================================
# 4. MODELS
# ============================================================

models = {

    "logistic_regression": Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}


# ============================================================
# 5. THRESHOLD CANDIDATES
# ============================================================

threshold_grid = np.arange(
    0.01,
    1.00,
    0.01,
)


# ============================================================
# 6. HELPER FUNCTION
# ============================================================

def positive_probability(model, X):

    return model.predict_proba(X)[:, 1]


# ============================================================
# 7. TRAIN / VALIDATION / TEST
# ============================================================

results = []

threshold_details = []


for horizon in FORECAST_HORIZONS:

    print()
    print("=" * 70)
    print(f"HORIZON +{horizon}h")
    print("=" * 70)

    target_column = (
        f"target_{horizon}h"
    )

    eligibility_column = (
        f"eligible_final_{horizon}h"
    )


    eligible = df[
        df[eligibility_column]
    ].copy()


    development_train = eligible[
        eligible.index
        < VALIDATION_START
    ]

    validation = eligible[
        (eligible.index >= VALIDATION_START)
        &
        (eligible.index < TEST_START)
    ]

    final_train = eligible[
        eligible.index
        < TEST_START
    ]

    test = eligible[
        eligible.index
        >= TEST_START
    ]


    X_dev = development_train[
        FEATURE_COLUMNS
    ]

    y_dev = (
        development_train[
            target_column
        ]
        .astype(int)
    )


    X_validation = validation[
        FEATURE_COLUMNS
    ]

    y_validation = (
        validation[
            target_column
        ]
        .astype(int)
    )


    X_final_train = final_train[
        FEATURE_COLUMNS
    ]

    y_final_train = (
        final_train[
            target_column
        ]
        .astype(int)
    )


    X_test = test[
        FEATURE_COLUMNS
    ]

    y_test = (
        test[
            target_column
        ]
        .astype(int)
    )


    print(
        "Development train:",
        len(X_dev),
        "| positives:",
        int(y_dev.sum())
    )

    print(
        "Validation:",
        len(X_validation),
        "| positives:",
        int(y_validation.sum())
    )

    print(
        "Final train:",
        len(X_final_train),
        "| positives:",
        int(y_final_train.sum())
    )

    print(
        "Test:",
        len(X_test),
        "| positives:",
        int(y_test.sum())
    )


    for model_name, model_template in models.items():

        # ====================================================
        # A. FIT USING 2018–2019
        # ====================================================

        validation_model = clone(
            model_template
        )

        validation_model.fit(
            X_dev,
            y_dev,
        )


        validation_score = (
            positive_probability(
                validation_model,
                X_validation,
            )
        )


        # ====================================================
        # B. SELECT THRESHOLD USING 2020 ONLY
        # ====================================================

        threshold_rows = []


        for threshold in threshold_grid:

            validation_pred = (
                validation_score
                >= threshold
            ).astype(int)


            precision = precision_score(
                y_validation,
                validation_pred,
                zero_division=0,
            )

            recall = recall_score(
                y_validation,
                validation_pred,
                zero_division=0,
            )

            f1 = f1_score(
                y_validation,
                validation_pred,
                zero_division=0,
            )


            threshold_rows.append(
                {
                    "threshold":
                        threshold,

                    "precision":
                        precision,

                    "recall":
                        recall,

                    "f1":
                        f1,
                }
            )


        threshold_df = pd.DataFrame(
            threshold_rows
        )


        best_row = (
            threshold_df
            .sort_values(
                [
                    "f1",
                    "recall",
                    "precision",
                ],
                ascending=False,
            )
            .iloc[0]
        )


        best_threshold = float(
            best_row["threshold"]
        )


        print()
        print(model_name)

        print(
            "Selected threshold:",
            round(
                best_threshold,
                3,
            )
        )

        print(
            "Validation precision:",
            round(
                best_row["precision"],
                3,
            )
        )

        print(
            "Validation recall:",
            round(
                best_row["recall"],
                3,
            )
        )

        print(
            "Validation F1:",
            round(
                best_row["f1"],
                3,
            )
        )


        threshold_details.append(
            {
                "horizon_hours":
                    horizon,

                "model":
                    model_name,

                "selected_threshold":
                    best_threshold,

                "validation_precision":
                    best_row["precision"],

                "validation_recall":
                    best_row["recall"],

                "validation_f1":
                    best_row["f1"],
            }
        )


        # ====================================================
        # C. RETRAIN USING 2018–2020
        # ====================================================

        final_model = clone(
            model_template
        )

        final_model.fit(
            X_final_train,
            y_final_train,
        )


        # ====================================================
        # D. FINAL 2021 TEST
        # ====================================================

        test_score = (
            positive_probability(
                final_model,
                X_test,
            )
        )


        test_pred = (
            test_score
            >= best_threshold
        ).astype(int)


        precision = precision_score(
            y_test,
            test_pred,
            zero_division=0,
        )

        recall = recall_score(
            y_test,
            test_pred,
            zero_division=0,
        )

        f1 = f1_score(
            y_test,
            test_pred,
            zero_division=0,
        )


        average_precision = (
            average_precision_score(
                y_test,
                test_score,
            )
        )


        roc_auc = roc_auc_score(
            y_test,
            test_score,
        )


        tn, fp, fn, tp = (
            confusion_matrix(
                y_test,
                test_pred,
                labels=[0, 1],
            )
            .ravel()
        )


        results.append(
            {
                "horizon_hours":
                    horizon,

                "model":
                    model_name,

                "threshold":
                    best_threshold,

                "precision":
                    precision,

                "recall":
                    recall,

                "f1":
                    f1,

                "average_precision":
                    average_precision,

                "roc_auc":
                    roc_auc,

                "true_negatives":
                    int(tn),

                "false_positives":
                    int(fp),

                "false_negatives":
                    int(fn),

                "true_positives":
                    int(tp),
            }
        )


        print()
        print("2021 TEST")

        print(
            f"Precision: {precision:.3f}"
        )

        print(
            f"Recall:    {recall:.3f}"
        )

        print(
            f"F1:        {f1:.3f}"
        )

        print(
            f"PR-AUC:    {average_precision:.3f}"
        )

        print(
            f"ROC-AUC:   {roc_auc:.3f}"
        )

        print(
            "Confusion matrix:",
            f"TN={tn}",
            f"FP={fp}",
            f"FN={fn}",
            f"TP={tp}",
        )


# ============================================================
# 8. FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)

threshold_details_df = pd.DataFrame(
    threshold_details
)


print()
print(
    "=== SELECTED VALIDATION THRESHOLDS ==="
)

display(
    threshold_details_df.round(3)
)


print()
print(
    "=== FINAL 2021 TEST RESULTS ==="
)

display(
    results_df[
        [
            "horizon_hours",
            "model",
            "threshold",
            "precision",
            "recall",
            "f1",
            "average_precision",
            "roc_auc",
            "true_positives",
            "false_positives",
            "false_negatives",
        ]
    ].round(3)
)


# ============================================================
# 9. F1 COMPARISON
# ============================================================

f1_comparison = (
    results_df
    .pivot(
        index="horizon_hours",
        columns="model",
        values="f1",
    )
)


print()
print(
    "=== FINAL F1 BY HORIZON ==="
)

display(
    f1_comparison.round(3)
)


# ============================================================
# 10. PRECISION COMPARISON
# ============================================================

precision_comparison = (
    results_df
    .pivot(
        index="horizon_hours",
        columns="model",
        values="precision",
    )
)


print()
print(
    "=== FINAL PRECISION BY HORIZON ==="
)

display(
    precision_comparison.round(3)
)


# ============================================================
# 11. RECALL COMPARISON
# ============================================================

recall_comparison = (
    results_df
    .pivot(
        index="horizon_hours",
        columns="model",
        values="recall",
    )
)


print()
print(
    "=== FINAL RECALL BY HORIZON ==="
)

display(
    recall_comparison.round(3)
)


# ============================================================
# 12. SAVE
# ============================================================

results_df.to_csv(
    OUTPUT_PATH,
    index=False,
)


print()
print(
    "Optimized results saved to:"
)

print(
    OUTPUT_PATH
)

Dataset loaded.
Rows: 33871

HORIZON +1h
Development train: 16860 | positives: 48
Validation: 8726 | positives: 22
Final train: 25586 | positives: 70
Test: 7248 | positives: 38

logistic_regression
Selected threshold: 0.95
Validation precision: 0.049
Validation recall: 0.409
Validation F1: 0.088

2021 TEST
Precision: 0.049
Recall:    0.711
F1:        0.092
PR-AUC:    0.206
ROC-AUC:   0.925
Confusion matrix: TN=6688 FP=522 FN=11 TP=27

random_forest
Selected threshold: 0.06
Validation precision: 0.035
Validation recall: 0.136
Validation F1: 0.056

2021 TEST
Precision: 0.131
Recall:    0.211
F1:        0.162
PR-AUC:    0.116
ROC-AUC:   0.891
Confusion matrix: TN=7157 FP=53 FN=30 TP=8

HORIZON +2h
Development train: 16845 | positives: 48
Validation: 8726 | positives: 22
Final train: 25571 | positives: 70
Test: 7246 | positives: 38

logistic_regression
Selected threshold: 0.94
Validation precision: 0.02
Validation recall: 0.182
Validation F1: 0.036

2021 TEST
Precision: 0.042
Recall:    0.

,horizon_hours,model,selected_threshold,validation_precision,validation_recall,validation_f1
0,1,logistic_regression,0.95,0.049,0.409,0.088
1,1,random_forest,0.06,0.035,0.136,0.056
2,2,logistic_regression,0.94,0.020,0.182,0.036
3,2,random_forest,0.05,0.017,0.091,0.028
4,3,logistic_regression,0.95,0.016,0.136,0.029
5,3,random_forest,0.05,0.026,0.136,0.043
6,4,logistic_regression,0.96,0.019,0.091,0.032
7,4,random_forest,0.02,0.018,0.318,0.035
8,5,logistic_regression,0.99,0.048,0.045,0.047
9,5,random_forest,0.04,0.018,0.136,0.032



=== FINAL 2021 TEST RESULTS ===


,horizon_hours,model,threshold,precision,recall,f1,average_precision,roc_auc,true_positives,false_positives,false_negatives
0,1,logistic_regression,0.95,0.049,0.711,0.092,0.206,0.925,27,522,11
1,1,random_forest,0.06,0.131,0.211,0.162,0.116,0.891,8,53,30
2,2,logistic_regression,0.94,0.042,0.605,0.078,0.078,0.888,23,525,15
3,2,random_forest,0.05,0.075,0.132,0.095,0.048,0.742,5,62,33
4,3,logistic_regression,0.95,0.049,0.421,0.088,0.064,0.881,16,310,22
5,3,random_forest,0.05,0.028,0.053,0.036,0.023,0.748,2,70,36
6,4,logistic_regression,0.96,0.084,0.342,0.135,0.068,0.885,13,142,25
7,4,random_forest,0.02,0.012,0.079,0.021,0.017,0.641,3,250,35
8,5,logistic_regression,0.99,0.106,0.132,0.118,0.044,0.850,5,42,33
9,5,random_forest,0.04,0.011,0.026,0.015,0.009,0.645,1,93,37



=== FINAL F1 BY HORIZON ===


...


=== FINAL PRECISION BY HORIZON ===


model,logistic_regression,random_forest
horizon_hours,,
1,0.049,0.131
2,0.042,0.075
3,0.049,0.028
4,0.084,0.012
5,0.106,0.011
6,0.034,0.000



=== FINAL RECALL BY HORIZON ===


model,logistic_regression,random_forest
horizon_hours,,
1,0.711,0.211
2,0.605,0.132
3,0.421,0.053
4,0.342,0.079
5,0.132,0.026
6,0.368,0.000



Optimized results saved to:
..\data\processed\threshold_optimized_results.csv


## Conclusion

Decision-threshold optimization substantially changed the behavior of the baseline classifiers.

The +1h horizon produced the strongest predictive performance.

Random Forest achieved the highest F1-score at +1h (0.162), with a precision of 0.131 and recall of 0.211.

Logistic Regression demonstrated substantially stronger event-ranking capability at the same horizon, reaching an average precision of 0.206 and ROC-AUC of 0.925. Its optimized threshold detected 27 of the 38 severe-rainfall observations in the 2021 test period, corresponding to a recall of 0.711, although at the cost of a large number of false-positive alerts.

Performance generally decreased as the forecasting horizon increased, reinforcing the difficulty of predicting severe rainfall several hours in advance using measurements from a single meteorological station.

The results also demonstrate that the conventional probability threshold of 0.5 is inappropriate for this highly imbalanced classification problem.

Both Logistic Regression and Random Forest remain candidates for final model development. The next stage will evaluate a limited set of hyperparameters using only pre-2021 temporal validation data.